# TrackSquad — Asset Risk ML Pipeline (2026 Refresh)
### Automatic Railway Block Planning — Offline Training Notebook

This notebook trains, compares, and selects the `asset_risk_score` regression
model used by the backend's `ml/inference.py`, on the **new ~65,000-row
datasets**. It reproduces exactly what was run to build the artifacts already
in `backend/app/ml/models/`.

**Rule enforced throughout:** only the four new datasets are used. Old/small
datasets are never loaded here.

Runs top-to-bottom on Google Colab with no hidden manual steps. Upload the
four CSVs when prompted in Step 3, or mount Drive and edit the paths.

In [1]:
# Step 1: Install dependencies
!pip -q install scikit-learn==1.8.0 xgboost pandas numpy joblib tabulate

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
# Step 2: Imports
import json, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, xgboost, joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

scikit-learn: 1.8.0 | xgboost: 3.4.1


## Step 3: Load the FOUR NEW datasets

If running on Colab, upload the four CSVs when prompted:
- `asset_health_dataset_65000.csv`
- `block_request_clean_expanded_60000.csv`
- `existing_blocks_dataset_65k_ref.csv`
- `isl_wise_train_detail_03082015_v1.csv`

**Old/small datasets from earlier project phases are never loaded in this
notebook — new-dataset-only, per the project rule.**

In [3]:
# Step 3: Load datasets (Colab upload widget; skip if files already local)
try:
    from google.colab import files
    print("Upload the four CSVs now (multi-select supported):")
    uploaded = files.upload()
except ImportError:
    uploaded = None  # not running on Colab -- assume files are already local

asset_health = pd.read_csv("asset_health_dataset_65000.csv")
block_request = pd.read_csv("block_request_clean_expanded_60000.csv")
existing_blocks = pd.read_csv("existing_blocks_dataset_65k_ref.csv")
train_detail = pd.read_csv("isl_wise_train_detail_03082015_v1.csv")

for name, df in [("asset_health", asset_health), ("block_request", block_request),
                  ("existing_blocks", existing_blocks), ("train_detail", train_detail)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols")

asset_health: 65000 rows x 12 cols
block_request: 60000 rows x 16 cols
existing_blocks: 65000 rows x 13 cols
train_detail: 69006 rows x 12 cols


## Step 4: Dataset inspection

Per-dataset shape, dtypes, missing values, duplicates, unique keys.

In [4]:
for name, df in [("asset_health", asset_health), ("block_request", block_request),
                  ("existing_blocks", existing_blocks), ("train_detail", train_detail)]:
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(df.dtypes)
    print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
    print("Duplicate rows:", df.duplicated().sum())
    print()

asset_health
asset_id                           str
asset_type                         str
section_id                         str
nearest_station_code               str
age_years                        int64
condition_score                float64
failure_count_24m                int64
days_since_last_maintenance      int64
usage_percent                    int64
criticality                        str
asset_risk_score               float64
maintenance_priority               str
dtype: object
Missing values:
 Series([], dtype: int64)
Duplicate rows: 0

block_request
block_request_id            str
asset_id                    str
section_id                  str
station_code                str
maintenance_type            str
requested_duration_min    int64
priority                    str
preferred_start_time        str
time_flexibility            str
required_team               str
request_urgency             str
status                      str
preferred_start_min       int64
flexibility_mi

Missing values:
 linked_block_request_id    3279
dtype: int64
Duplicate rows: 0

train_detail
Train No.                     str
train Name                    str
islno                       int64
station Code                  str
Station Name                  str
Arrival time                  str
Departure time                str
Distance                    int64
Source Station Code           str
source Station Name           str
Destination station Code      str
Destination Station Name      str
dtype: object
Missing values:
 Series([], dtype: int64)
Duplicate rows: 0



## Step 5: Data validation — cross-dataset relationship audit

This is the most important check before deciding what to join. We verify
`asset_id` overlap (the one reliable key across datasets), and separately
check whether `existing_blocks`' own `section_id`/`station_code` columns
agree with the authoritative per-asset location recorded in `asset_health`.

**Finding (documented in the chat-turn audit):** `existing_blocks`' own
`section_id`/`station_code` do NOT reliably match asset_health for the same
`asset_id` (>99.9% disagreement) because the datasets were expanded
independently. `asset_id` itself IS 100% valid across all three operational
datasets. `block_request`'s own `section_id`/`station_code` DO already match
asset_health correctly (0% disagreement) — no correction needed there.

In [5]:
ah_lookup = asset_health.set_index("asset_id")[["section_id", "nearest_station_code"]]

# block_request: already consistent (spot-check)
br_check = block_request.merge(ah_lookup, left_on="asset_id", right_index=True,
                                suffixes=("_br", "_ah"))
print("block_request section_id matches asset_health:",
      (br_check.section_id_br == br_check.section_id_ah).mean())
print("block_request station_code matches asset_health:",
      (br_check.station_code == br_check.nearest_station_code).mean())

# existing_blocks: NOT consistent -- this is why we correct it below
eb_check = existing_blocks.merge(ah_lookup, left_on="asset_id", right_index=True,
                                  suffixes=("_eb", "_ah"))
print("\nexisting_blocks section_id matches asset_health:",
      (eb_check.section_id_eb == eb_check.section_id_ah).mean())
print("existing_blocks station_code matches asset_health:",
      (eb_check.station_code == eb_check.nearest_station_code).mean())

print("\nasset_id overlap (the one reliable key):")
print("  block_request.asset_id in asset_health:", block_request.asset_id.isin(asset_health.asset_id).mean())
print("  existing_blocks.asset_id in asset_health:", existing_blocks.asset_id.isin(asset_health.asset_id).mean())

block_request section_id matches asset_health: 1.0
block_request station_code matches asset_health: 1.0

existing_blocks section_id matches asset_health: 0.0005384615384615384
existing_blocks station_code matches asset_health: 0.0007230769230769231

asset_id overlap (the one reliable key):
  block_request.asset_id in asset_health: 1.0
  existing_blocks.asset_id in asset_health: 1.0


## Step 6: Data cleaning / correction

- `asset_health`: used as-is (self-contained, already has the ground-truth
  target column and passed range/missing-value checks).
- `block_request`: used as-is (own location fields already correct).
- `existing_blocks`: **corrected** — `section_id`/`station_code` are
  re-derived from `asset_id` → `asset_health` (the reliable key), since the
  dataset's own values are unreliable at this scale (see Step 5).
  `linked_block_request_id` is nulled (only 0.7% of values coincidentally
  matched a real block_request_id — not a real link).

This correction affects the **runtime constraint-checking data**, not the ML
training data below (asset_health needs no correction and no join at all).

In [6]:
existing_blocks_corrected = existing_blocks.copy()
existing_blocks_corrected["raw_section_id_unreliable"] = existing_blocks_corrected["section_id"]
existing_blocks_corrected["raw_station_code_unreliable"] = existing_blocks_corrected["station_code"]
existing_blocks_corrected = existing_blocks_corrected.drop(columns=["section_id", "station_code"]).merge(
    ah_lookup, left_on="asset_id", right_index=True, how="left"
).rename(columns={"nearest_station_code": "station_code"})
existing_blocks_corrected["linked_block_request_id"] = pd.NA

assert existing_blocks_corrected["section_id"].isna().sum() == 0
print("existing_blocks corrected:", len(existing_blocks_corrected), "rows")
existing_blocks_corrected.head(3)

existing_blocks corrected: 65000 rows


,existing_block_id,linked_block_request_id,asset_id,block_type,start_time,end_time,duration_min,assigned_team,status,operational_priority,source,raw_section_id_unreliable,raw_station_code_unreliable,section_id,station_code
0,EB-0001,<NA>,AST-0063,Engineering Block,18:42,19:27,45,Signal Team,Confirmed,High,Existing/Committed,SEC-019,MKU,SEC-019,MKU
1,EB-0002,<NA>,AST-0049,Maintenance Block,00:47,01:32,45,OHE Team,Confirmed,Normal,Existing/Committed,SEC-025,LTRR,SEC-025,LTRR
2,EB-0003,<NA>,AST-0122,Engineering Block,10:08,10:38,30,Track Maintenance Team,Active,Normal,Existing/Committed,SEC-042,HBD,SEC-042,HBD


## Step 7: Determine the ML problem

`asset_health` already contains the ground-truth `asset_risk_score` column
(0-100, continuous) for every one of its 65,000 rows. **No target needs to be
invented.** This is a **regression** problem, exactly as in the original
(200-row) version of this project — just with far more data.

`maintenance_priority` is excluded from the features: it matches the
score-cutoff bucket 85% of the time in this dataset, i.e. it is derived from
(or very close to) the target, so using it as a feature would leak the
answer.

`block_request` and `existing_blocks` are **not** used in training — they
never were (even in the original 200-row version); they exist to feed the
candidate generator / constraint engine at request time, not the ML model.

In [7]:
TARGET = "asset_risk_score"
NUMERIC_FEATURES = ["age_years", "condition_score", "failure_count_24m",
                     "days_since_last_maintenance", "usage_percent"]
CATEGORICAL_FEATURES = ["criticality", "asset_type"]
EXCLUDED = ["asset_id", "section_id", "nearest_station_code", "maintenance_priority", TARGET]

X = asset_health[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = asset_health[TARGET]

# Sanity: is the target a simple deterministic formula of the numeric features?
# (If R^2 were ~1.0 here we'd be worried about a trivial/leaked target.)
lr_check = LinearRegression().fit(X[NUMERIC_FEATURES], y)
print("Linear fit on numeric features only, R^2:", lr_check.score(X[NUMERIC_FEATURES], y))
print("-> meaningfully below 1.0: real signal + noise, not a disguised formula.")

print("\nmaintenance_priority vs score-bucket agreement (why it's excluded):")
def bucket(s):
    if s < 35: return "Low"
    elif s < 60: return "Medium"
    elif s < 80: return "High"
    else: return "Critical"
print((asset_health[TARGET].apply(bucket) == asset_health["maintenance_priority"]).mean())

Linear fit on numeric features only, R^2: 0.30792570768737204
-> meaningfully below 1.0: real signal + noise, not a disguised formula.

maintenance_priority vs score-bucket agreement (why it's excluded):
0.8528461538461538


## Step 8: Feature engineering

| Feature | Source | Meaning | Why useful |
|---|---|---|---|
| age_years | asset_health | Age of the asset in years | Older assets generally carry more risk |
| condition_score | asset_health | Physical condition, 0-100 | Direct condition signal, strongest single numeric feature |
| failure_count_24m | asset_health | Failures in last 24 months | Recent failure history predicts future risk |
| days_since_last_maintenance | asset_health | Maintenance recency | Longer gaps correlate with higher risk |
| usage_percent | asset_health | Utilization | Higher usage increases wear |
| criticality | asset_health | Operational importance (Low/Med/High/Critical) | Dominant categorical driver of risk in this dataset (see feature importances below) |
| asset_type | asset_health | Track/Bridge/Signal/OHE/Point Machine/Level Crossing | Different asset classes have different risk profiles |

No features are engineered from `block_request` / `existing_blocks` /
`train_detail` — they have no valid, non-fabricated relationship to
per-asset risk (see Step 5).

## Step 9: Target definition (recap)
Target: `asset_risk_score` — regression, 0-100, present directly in the
data. See Step 7.

## Step 10: Train / validation / test split

60/20/20, stratified on `criticality`. Not time-series data (no date/time
column in asset_health; each row is an independent asset), so a stratified
random split is appropriate. Verified below that no row index appears in
more than one split (no data leakage).

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=X["criticality"])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=X_temp["criticality"])

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
assert not (set(X_train.index) & set(X_val.index) & set(X_test.index))
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0
print("No index overlap across splits.")

Train: 39000 | Val: 13000 | Test: 13000
No index overlap across splits.


## Step 11: Preprocessing pipeline

Numeric features pass through unchanged; categorical features are
one-hot encoded (unknown categories at inference time are handled
gracefully, not rejected). This preprocessing is saved INSIDE the final
pipeline artifact — inference never needs a separate preprocessing step.

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

## Step 12: Train multiple models

Baseline (Linear Regression) plus three stronger nonlinear/ensemble
candidates.

In [10]:
candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=10,
                                           min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_depth=6, learning_rate=0.08,
                                                           max_iter=300, random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                             subsample=0.9, colsample_bytree=0.9,
                             random_state=RANDOM_STATE, n_jobs=-1),
}

def evaluate(model, X_eval, y_eval):
    t0 = time.perf_counter()
    preds = model.predict(X_eval)
    batch_ms = (time.perf_counter() - t0) * 1000
    t1 = time.perf_counter()
    _ = model.predict(X_eval.iloc[[0]])
    single_ms = (time.perf_counter() - t1) * 1000
    return {"rmse": float(np.sqrt(mean_squared_error(y_eval, preds))),
            "mae": float(mean_absolute_error(y_eval, preds)),
            "r2": float(r2_score(y_eval, preds)),
            "batch_inference_ms_total": batch_ms,
            "single_prediction_latency_ms": single_ms}, preds

results, fitted = {}, {}
for name, estimator in candidates.items():
    pipe = Pipeline([("preprocess", preprocessor), ("regressor", estimator)])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time_s = time.perf_counter() - t0
    metrics, _ = evaluate(pipe, X_val, y_val)
    metrics["train_time_s"] = train_time_s
    results[name] = metrics
    fitted[name] = pipe
    print(f"{name:22s} val RMSE={metrics['rmse']:.3f} MAE={metrics['mae']:.3f} "
          f"R2={metrics['r2']:.4f} train_time={train_time_s:.2f}s")

LinearRegression       val RMSE=5.026 MAE=3.989 R2=0.8783 train_time=0.06s


RandomForest           val RMSE=5.166 MAE=4.093 R2=0.8714 train_time=22.00s


HistGradientBoosting   val RMSE=5.078 MAE=4.033 R2=0.8758 train_time=0.45s


XGBoost                val RMSE=5.098 MAE=4.043 R2=0.8748 train_time=0.96s


## Step 13: Model comparison table (real validation results)

In [11]:
comparison_df = pd.DataFrame(results).T
comparison_df.index.name = "model"
comparison_df

,rmse,mae,r2,batch_inference_ms_total,single_prediction_latency_ms,train_time_s
model,,,,,,
LinearRegression,5.025507,3.988563,0.878334,5.924201,2.519370,0.057424
RandomForest,5.166389,4.093234,0.871417,353.205005,18.248325,21.998360
HistGradientBoosting,5.078292,4.033053,0.875765,56.261204,3.069532,0.445523
XGBoost,5.098480,4.042932,0.874775,65.247209,2.968329,0.963246


## Step 14: Best model selection

Selected on **lowest validation RMSE**, not training accuracy. All
candidates have sub-5ms single-prediction latency, so speed does not
eliminate any candidate here — selection is driven by generalization,
with simplicity/explainability as a tie-breaker when performance is close.

In [12]:
best_name = min(results, key=lambda k: results[k]["rmse"])
best_pipeline = fitted[best_name]
print("Selected model:", best_name)
print(f"Validation RMSE: {results[best_name]['rmse']:.3f}")

Selected model: LinearRegression
Validation RMSE: 5.026


## Step 15: Final evaluation on the untouched TEST set

The test set was not used anywhere above (not for training, not for model
selection) — this is the one honest, final number.

In [13]:
test_metrics, test_preds = evaluate(best_pipeline, X_test, y_test)
print("Test RMSE:", round(test_metrics["rmse"], 3))
print("Test MAE:", round(test_metrics["mae"], 3))
print("Test R2:", round(test_metrics["r2"], 4))

def bucket_priority(score):
    if score < 35: return "Low"
    elif score < 60: return "Medium"
    elif score < 80: return "High"
    else: return "Critical"

pred_priority = pd.Series(test_preds, index=X_test.index).apply(bucket_priority)
true_priority = asset_health.loc[X_test.index, "maintenance_priority"]
bucket_acc = (pred_priority == true_priority).mean()
print(f"Downstream bucket accuracy: {bucket_acc:.3f}")

feature_names = (NUMERIC_FEATURES +
    list(best_pipeline.named_steps["preprocess"].named_transformers_["cat"]
         .get_feature_names_out(CATEGORICAL_FEATURES)))
reg = best_pipeline.named_steps["regressor"]
if hasattr(reg, "feature_importances_"):
    imp = pd.DataFrame({"feature": feature_names, "importance": reg.feature_importances_})
elif hasattr(reg, "coef_"):
    imp = pd.DataFrame({"feature": feature_names, "importance": np.abs(reg.coef_)})
else:
    imp = pd.DataFrame({"feature": feature_names, "importance": np.nan})
imp.sort_values("importance", ascending=False)

Test RMSE: 5.073
Test MAE: 4.043
Test R2: 0.8742
Downstream bucket accuracy: 0.748


,feature,importance
5,criticality_Critical,17.980377
7,criticality_Low,17.062362
8,criticality_Medium,6.950780
6,criticality_High,6.032764
2,failure_count_24m,1.115105
1,condition_score,0.301987
10,asset_type_Level Crossing,0.109441
0,age_years,0.074182
12,asset_type_Point Machine,0.063047
4,usage_percent,0.061151


## Step 16: Inference timing (offline training vs online inference)

The key architectural point for the mentor's concern: the model is trained
**once, here, offline**. The backend only ever calls `.predict()` on the
already-loaded pipeline.

In [14]:
n_reps = 200
sample_row = X_test.iloc[[0]]
t0 = time.perf_counter()
for _ in range(n_reps):
    best_pipeline.predict(sample_row)
avg_ms = (time.perf_counter() - t0) / n_reps * 1000
print(f"Average single-prediction latency over {n_reps} calls: {avg_ms:.3f} ms")

Average single-prediction latency over 200 calls: 1.942 ms


## Step 17: Save the complete pipeline (preprocessing + model)

In [15]:
joblib.dump(best_pipeline, "asset_risk_model.joblib")
print("Saved asset_risk_model.joblib")

Saved asset_risk_model.joblib


## Step 18: Save metadata

In [16]:
metadata = {
    "model_name": best_name,
    "model_version": "2.0.0",
    "target": TARGET,
    "feature_names_raw": NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    "feature_names_encoded": feature_names,
    "excluded_leakage_fields": EXCLUDED,
    "dataset_name": "asset_health_dataset_65000 (2026 refresh)",
    "final_training_record_count": len(asset_health),
    "split_sizes": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "split_strategy": "random, stratified on criticality (60/20/20)",
    "validation_metrics_all_candidates": results,
    "final_test_metrics": {**test_metrics, "downstream_bucket_accuracy": float(bucket_acc)},
    "feature_importance": imp.to_dict(orient="records"),
    "training_date": pd.Timestamp.now().isoformat(),
    "library_versions": {"python": platform.python_version(), "scikit_learn": sklearn.__version__,
                          "xgboost": xgboost.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "honesty_note": ("asset_health_dataset_65000.csv is a simulated/synthetic railway asset "
                      "dataset, not real IR sensor telemetry. This is an ML-based asset risk "
                      "estimation prototype, not a production failure-prediction system."),
}
with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print("Saved model_metadata.json")

Saved model_metadata.json


## Step 19: Download artifacts (Colab)

In [17]:
try:
    from google.colab import files
    files.download("asset_risk_model.joblib")
    files.download("model_metadata.json")
except ImportError:
    print("Not running on Colab -- files are already saved locally:")
    print(" - asset_risk_model.joblib")
    print(" - model_metadata.json")

Not running on Colab -- files are already saved locally:
 - asset_risk_model.joblib
 - model_metadata.json
